# Telecommunications Project

The telecom operator TeleDom wants to reduce customer churn. If we can detect in advance that a subscriber is likely to terminate the contract, we can offer a promo code, a discount, or special terms in time, and retain some of those customers.

In this project I will build a machine learning model that predicts **whether a subscriber will terminate the contract or stay**. This is a classification task.

I have data from several sources (different files) that must be carefully merged by the customerID key:

- contract_new.csv - contract and billing details: start/end dates, contract type, payment method, monthly and total charges, etc.
- personal_new.csv - basic customer attributes: gender, senior citizen status, partner, dependents.
- internet_new.csv - internet and add-on services: connection type (DSL/Fiber optic), security, backups, tech support, streaming services.
- phone_new.csv - telephony: for example, multi-line service.

Important: the contract data is current as of Feb 1, 2020 - this is the snapshot date used to create the target and build the forecast.

-----

Work plan:

 1) Data loading and initial inspection
- Load all tables and quickly check sizes, data types, missing values, duplicates.
- Verify the customerID key and whether customers match across files.

 2) Analysis and preprocessing of each dataset
- Normalize formats (especially dates and monetary fields).
- Handle missing values and odd entries.
- Assess which features will be useful and which may be noisy or problematic.

 3) Data merging
- Build a single dataframe by customerID.
- Check if any customers are lost during merging and why (if so).

 4) EDA and preprocessing of the combined dataset
- Inspect feature distributions and their relationship with churn.
- Run correlation analysis to detect strong dependencies/duplicates.
- Create new features when useful, for example:
  - contract duration (from dates),
  - number of connected services,
  - service bundles (e.g., streaming/security/tech support as a group).

 5) Prepare data for training
- Split data into train/test.
- Set up preprocessing (categorical encoding, scaling of numeric features where needed) without leakage from test to train.
- Build a clean pipeline for reproducibility.

 6) Model training
- Train at least two models and compare their quality.
- For one model, tune at least two hyperparameters to improve the result.

 7) Select the best model and final check
- Choose the best model by quality and evaluate on the test set.
- Record the final metric and key error analysis (where the model fails and why).

 8) Final conclusion and recommendations
- Summarize what was done and the results.
- Provide business recommendations: which customer groups to focus on, which features best signal churn risk, and how to use the model for retention.

## Data Loading

In [ ]:
pip install phik

In [ ]:
!pip install shap

In [ ]:
pip install -U scikit-learn

In [ ]:
pip install -U lightgbm

In [ ]:
# import required modules and libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from phik import phik_matrix
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RandomizedSearchCV, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier
import shap
import warnings

In [ ]:
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
# read the available datasets
contract = pd.read_csv('/datasets/contract_new.csv')  # contract information
personal = pd.read_csv('/datasets/personal_new.csv')  # customer's personal data
internet = pd.read_csv('/datasets/internet_new.csv')  # internet services information
phone = pd.read_csv('/datasets/phone_new.csv')  # telephony services information

In [ ]:
contract.head(10)

In [ ]:
personal.head(10)

In [ ]:
internet.head(10)

In [ ]:
phone.head(10)

In [ ]:
contract.info()

In [ ]:
personal.info()

In [ ]:
internet.info()

In [ ]:
phone.info()

### Summary:

During the initial inspection I reviewed the dataset columns and their meanings. The datasets have no missing values. The 'contract' dataset needs data type adjustments.

## Data Preprocessing and EDA of Individual Datasets

### Data Type Changes

In [ ]:
# change the 'BeginDate' column type to datetime
contract['BeginDate'] = pd.to_datetime(contract['BeginDate'])

It is not possible to change the 'EndDate' column type right now because it contains the value "No" for active customers. This is a good time to create the target feature 'IsBrokenContract', where 1 means the contract is terminated and 0 means it is active.

In [ ]:
contract['IsBrokenContract'] = (contract['EndDate'] != "No").astype(int)

Now we can convert the 'EndDate' column to datetime using errors='coerce'.

In [ ]:
contract['EndDate'] = pd.to_datetime(contract['EndDate'], errors='coerce')

In [ ]:
# change the 'MonthlyCharges' column type to float
contract['MonthlyCharges'] = contract['MonthlyCharges'].astype('float')

An error occurs when converting 'TotalCharges' because this column contains strings made of spaces. Let's check them.

In [ ]:
# count rows with an empty value
(contract['TotalCharges'].astype(str).str.strip() == "").sum()

In [ ]:
# show rows where 'TotalCharges' is empty
s = contract['TotalCharges'].astype(str)
mask = s.str.strip().eq('')
contract[mask]

All users in these rows started their contract on 2020-02-01, which is the snapshot date. Therefore, 'TotalCharges' has not accumulated yet for them, and the missing values should be 0.

In [ ]:
contract["TotalCharges"] = pd.to_numeric(contract["TotalCharges"], errors="coerce").fillna(0)

I converted the 'BeginDate' and 'EndDate' columns to datetime after creating the target feature 'IsBrokenContract', handled missing values in 'TotalCharges', and changed the 'TotalCharges' and 'MonthlyCharges' columns to float.

### Duplicate Check

Let's check for explicit duplicates in the 'customerID' column to avoid cases where a single user has multiple records. If there are no explicit duplicates in these columns, there should be none in the full dataset.

In [ ]:
contract['customerID'].duplicated().sum()

In [ ]:
personal['customerID'].duplicated().sum()

In [ ]:
internet['customerID'].duplicated().sum()

In [ ]:
phone['customerID'].duplicated().sum()

Let's check some dataset columns for implicit duplicates.

In [ ]:
# create a function to detect implicit duplicates
def check_duplicates(data):
    for column in data.select_dtypes(include='object'):
        if column != 'customerID':
            print(f"{column:<20}:  {data[column].unique()}")  

In [ ]:
check_duplicates(contract)

In [ ]:
check_duplicates(personal)

In [ ]:
check_duplicates(internet)

In [ ]:
check_duplicates(phone)

There are no explicit or implicit duplicates in the data.

### Preprocessing Summary:

During preprocessing, it turned out that the EndDate column could not be immediately converted to datetime because active customers have the value "No". Therefore, the target feature IsBrokenContract was created first (1 = terminated, 0 = active), after which EndDate was converted using errors='coerce'.

When converting TotalCharges to numeric, blank strings were found; checking showed these records belong to customers who started on 2020-02-01 (the snapshot date), so total charges had not accumulated yet and missing values were correctly set to 0. As a result, BeginDate and EndDate were converted to datetime, and MonthlyCharges and TotalCharges to float.

Additionally, duplicates were checked: explicit (by customerID) and implicit (via check_duplicates) - no duplicates were found.

### EDA of the 'contract' dataset

In [ ]:
# create a function for exploratory data analysis
def analyze_distribution(data, column):
    num_columns_data = data.select_dtypes(include='float').columns.to_list()
    name = (column
        .replace('_', ' ')
        .capitalize())
    
    if column in num_columns_data:
        data[column].hist(bins=20, color='#F5C6AA')
        plt.xlabel(name)
        plt.ylabel('Number of users')
        plt.title(f'Distribution of "{name}"')  # removed an extra quote
        plt.show()
        
        plt.boxplot(data[column])
        plt.xlabel(name)
        plt.ylabel('Number of users')
        plt.title(f'Distribution of "{name}"')  # removed an extra quote
        plt.show()

        print(data[column].describe())
    
    elif data[column].dtype == 'datetime64[ns]':
        print(name)
        print('Minimum date:', data[column].min(skipna=True))
        print('Maximum date:', data[column].max(skipna=True))
        
        
    
    else:
        counts = data[column].value_counts()
        counts.plot(kind='bar', color='#F5C6AA')
        plt.xlabel(name)
        plt.ylabel('Number of users')
        plt.title(f'Distribution of "{name}"')
        plt.show()
        plt.pie(
        counts,
        labels=counts.index,
        autopct='%.1f%%')
        plt.title(f'Percentage distribution of "{name}"')
        plt.show()

In [ ]:
for column in contract.drop('customerID', axis=1).columns:
    analyze_distribution(contract, column)


- **Time coverage:** contracts start on 2013-10-01 and are recorded up to 2020-02-01. End dates (EndDate) range from 2014-06-01 to 2020-01-01, so the sample covers several years of customer history.

- **Contract type:** month-to-month dominates at about 55% of customers. Long-term contracts are less common: Two year ~24.1%, One year ~20.9%. This suggests most customers prefer more flexible terms.

- **Paperless billing:** most customers use electronic bills (~59.2%) versus ~40.8% without paperless billing. Digital channels are important and widely adopted.

- **Payment method:** the most popular option is Electronic check (~33.6%). The rest are fairly even: Mailed check (22.9%), Bank transfer (automatic) (21.9%), Credit card (automatic) (21.6%).

**Monthly charges (MonthlyCharges):**
  - observations: 7043;
  - mean: ~64.76, median: ~70.35;
  - range: 18.25 - 118.75;
  - half of customers pay roughly in the 35.5 - 89.85 interval (IQR).
    The distribution is heterogeneous (groups with low and high payments), likely reflecting different services (internet and telephony).

**Total charges (TotalCharges):**
  - mean: ~2115.31, median: ~1343.35;
  - range: 0 - 9221.38;
  - the distribution is strongly right-skewed and contains notable outliers (customers with very large cumulative spending).

### EDA of the 'personal' dataset

In [ ]:
for column in personal.drop('customerID', axis=1).columns:
    analyze_distribution(personal, column)

- **Gender:** the distribution is almost even - Male ~50.5%, Female ~49.5%. There is no strong gender skew, so the feature can be used without concern about very small groups.

- **Senior citizen:** most customers are not seniors - 0: ~83.8%, while seniors (1) are ~16.2%. The feature is imbalanced, but 16% is large enough to analyze behavior differences (e.g., churn probability).

- **Partner:** the shares of customers with and without a partner are close, with a slight advantage for "No" at ~51.7% vs "Yes" at ~48.3%. This feature may be moderately informative; later it is worth testing whether customers with partners stay longer due to more stable households.

- **Dependents:** most customers have no dependents - No ~70%, Yes ~30%. The feature is skewed toward "No", which may reflect the customer base structure. Dependents could be linked to more stable service usage and different tariff behavior, so it should be considered in churn and payment analysis.

### EDA of the 'internet' dataset

In [ ]:
for column in internet.drop('customerID', axis=1).columns:
    analyze_distribution(internet, column)

- **Connection type:** Fiber optic dominates at about 56.1%, DSL is about 43.9%. Fiber is more common, but the distribution is fairly balanced.

- **Online security (blocking unsafe sites):** the service is used less often than not - No ~63.4%, Yes ~36.6%. This suggests most customers do not enable extra protection (possibly due to price or low perceived value).

- **Online backup:** closer to balance, but still more often not enabled - No ~56.0%, Yes ~44.0%. Backup is used by a substantial share.

- **Device protection/antivirus:** similar to OnlineBackup - No ~56.1%, Yes ~43.9%. Additional protection is popular but still a minority.

- **Tech support:** enabled relatively rarely - Yes ~37.0%, No ~63.0%. Many customers likely do not see the need or do not want to pay extra.

- **Streaming TV:** almost perfectly balanced - No ~50.9%, Yes ~49.1%. The service is widely used by about half of the customer base.

- **Streaming movies:** also near 50/50 - No ~50.5%, Yes ~49.5%. Video services are widely used and about half of customers have them.

Overall, for internet services: entertainment services (StreamingTV/Movies) are enabled by about half of customers, while protective/service options (OnlineSecurity, TechSupport) are noticeably less common at around one-third, which may reflect price sensitivity or lower perceived value.

### EDA of the 'phone' dataset

In [ ]:
for column in phone.drop('customerID', axis=1).columns:
    analyze_distribution(phone, column)

The distribution is nearly uniform, so this feature can be used for comparisons between groups.

### EDA Summary

The data describes customers and their contracts for 2013-2020. The target IsBrokenContract is imbalanced (about 15.6% churn), which is expected. There are strong behavioral and product features (contract type, payments, connected services) that are usually most useful for churn prediction. Some features are identifiers or duplicative/potentially leaking information and should be excluded from training.

Features that are useful and should be kept
1) **Contract and payment behavior**
- Type - reflects commitment level (month-to-month can be associated with churn).
- PaymentMethod - distinguishes autopay vs manual payments; manual payments can be more often associated with churn.
- PaperlessBilling - related to the two features above.
- MonthlyCharges - current plan cost; useful to inspect its relationship with churn.
- TotalCharges - indicates relationship duration and overall customer value.

2) **Product services (internet/services)**
- InternetService - connection type can be linked to price, quality, and churn.
- OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport - add-on services; help distinguish customers with bundled/high engagement. These services are less popular than entertainment services, and their relationship with churn is worth checking.
- StreamingTV, StreamingMovies - more popular services; may also affect churn.
- MultipleLines - telephony service bundle characteristic; may be useful.

3) **Socio-demographic features**
- SeniorCitizen - may show consistent behavioral differences.
- Partner, Dependents - household features; may be useful in combination.
- Gender - distribution is nearly even; it might be unimportant but should not be removed without checking correlations.

**Features better removed / not used in the model**
- customerID - identifier, not meaningful for prediction, can lead to overfitting.
- EndDate - effectively used to build IsBrokenContract (if EndDate has a date, the contract ended). This is target leakage: the model would "predict" churn from the event itself.
- BeginDate - not useful in raw form; better to create a duration feature.

## Data Merging

### Data Merging and New Feature Creation

In [ ]:
# put the contract dataframe into main
main = contract

In [ ]:
# create a list of the remaining dataframes
datas = [personal, internet, phone]

In [ ]:
# left-join each dataframe to main to keep all users
for data in datas:
    main = main.merge(data, on='customerID', how='left')

In [ ]:
# show the resulting dataframe
main.head()

In [ ]:
# check dataframe info after merging
main.info()

All data is present, and as expected there are missing values in service columns because not all users use all services. Data types are correct; the remaining tasks are to drop unnecessary columns and create a new feature: contract duration.

In [ ]:
# set customerID as the index
main = main.set_index('customerID')

In [ ]:
current_date = pd.to_datetime('2020-02-01')

In [ ]:
# add a new feature 'ContractDuration' - contract duration
main['ContractDuration'] = np.where(
    main['EndDate'].notna(),
    main['EndDate'] - main['BeginDate'], 
    current_date - main['BeginDate'])

In [ ]:
# convert the new feature to float
main['ContractDuration'] = (main['ContractDuration'].dt.total_seconds() // 86400).astype('float')

In [ ]:
main.head(10)

In [ ]:
# drop the EndDate and BeginDate columns
main = main.drop(['EndDate', 'BeginDate'], axis=1)

In [ ]:
# check dataframe info after dropping columns and creating the new feature
main.info()

### Missing Value Handling in the New DataFrame

Missing values in internet-service columns indicate that the user does not use the internet, and missing values in the 'MultipleLines' column indicate that the user does not use telephony. Therefore, there cannot be missing values simultaneously in internet columns and in the telephony column. Let's verify this.

In [ ]:
# check missing values in any internet column and in 'MultipleLines'
((main['OnlineSecurity'].isna()) & (main['MultipleLines'].isna())).sum()

Indeed, there are no users who do not use both internet and telephony at the same time. Therefore, missing values in internet-service columns can be filled with "No internet service", and in the telephony column with "No phone service".

In [ ]:
internet_columns = internet.drop('customerID', axis=1).columns

In [ ]:
internet_columns

In [ ]:
# fill missing values in internet service columns
main[internet_columns] = main[internet_columns].fillna('No internet service')

In [ ]:
# fill missing values in the telephony service column
main['MultipleLines'] = main['MultipleLines'].fillna('No phone service')

In [ ]:
main.info()

### Summary:

In this step I merged the data by the 'customerID' key and then moved it to the index. I created a new feature 'ContractDuration' (contract length). I removed 'BeginDate' (not informative by itself) and 'EndDate' (would cause target leakage).

I filled missing values in the merged dataset: missing values in service columns indicate the user does not use internet/telephony, so internet-service missing values were filled with 'No internet service', and the telephony column with 'No phone service'.

## EDA of the Combined DataFrame

### Exploratory Data Analysis

In [ ]:
# use the function for exploratory data analysis
for column in main.columns:
    analyze_distribution(main, column)

**Summary:** For the columns 'Type', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'IsBrokenContract', 'Gender', 'SeniorCitizen', 'Partner', 'Dependents', the distributions did not change; a brief recap:

Month-to-month billing dominates (~55%). About 59% use paperless billing, and the most popular payment method is electronic check (~33.6%). Monthly charges vary widely (mean ~64.8, median ~70.35, range 18.25-118.75), while total charges are right-skewed with outliers (up to ~9221), reflecting large differences in customer value and tenure. The target IsBrokenContract is imbalanced: churn is about 15.6%. Most users are non-seniors (83.8%) and without dependents (70%); gender and partner status are nearly even.

We also know that 21.7% of users do not use internet and 9.7% do not use telephony.
Internet-service distribution:
- 44% use fiber optic and 34.4% use DSL.
- Unsafe-site blocking is used by 28.7%; almost half do not use it, and tech support shows a similar pattern.
- Antivirus is used more often: 34.5% use it, 43.8% do not.
- Streaming TV is nearly balanced: 38.4% "yes" vs 39.9% "no".

Distribution for the "multiple lines" phone service:
- 42.2% use it, 48.1% do not.

From the new 'ContractDuration' feature, the base consists of several cohorts:
- many customers with tenure up to ~1 year (up to 276 days - 25% of the base);
- a large cohort around ~2 years (median 761 days ~ 2.1 years);
- a notable share of "older" customers 4+ years (75% = 1461 days ~ 4 years, max ~ 6.3 years).

The strong right skew means there are many short contracts and fewer long ones, though not rare. Therefore, the mean (~2.46 years) describes the "typical" customer worse than the median (~2.1 years).



It is also worth examining feature distributions for different target classes; we will write a new function for that.

In [ ]:
def analyze_class_distribution(data, column):
    num_columns_data = data.select_dtypes(include='float').columns.to_list()
    # for numeric columns, plot density distributions for classes 0 and 1
    if column in num_columns_data:
        plt.figure(figsize=(8,4))
        sns.histplot(
        data=data, x=column, hue='IsBrokenContract',
        bins=30,
        stat="density",
        common_norm=False,   
        multiple="layer",
        alpha=0.4
        )
        plt.title(f"{column}: density by class")
        plt.show()
    # for categorical columns, plot class shares within each category
    else:
        ct = pd.crosstab(data[column], data['IsBrokenContract'], normalize="index")  
        ax = ct.plot(kind="bar", stacked=True, figsize=(8,4))
        ax.set_ylabel("Share within category")
        ax.set_title(f"{column}: class shares within each category")
        plt.xticks(rotation=0)
        plt.tight_layout()
        plt.show()

In [ ]:
for column in main.drop('IsBrokenContract', axis=1).columns:
    analyze_class_distribution(main, column)

**Summary:**
Class shares within each category for categorical features are as follows:
- Contract type: the largest share is the two-year plan (~20%), with the one-year plan close behind; month-to-month users terminate about half as often.
- Customers receiving electronic checks terminate slightly more often (~5%).
- The lowest churn is among MailedCheck and ElectronicCheck users; automatic payment users churn more often.
- Men terminate slightly more often than women.
- Seniors churn more often.
- Customers with a partner churn almost twice as often as those without.
- Dependents have a small effect on churn.
- Fiber optic internet is associated with churn about 5-7% more often.
- Other internet and telephony services also increase churn likelihood, which may be tied to service dissatisfaction.

From density distributions of numeric features:
- Higher monthly charges correspond to higher churn probability.
- Churn probability is higher for total charges between 2000 and 4000; after 4000 it drops sharply and reaches near 0 above 8000, indicating the most loyal users.
- For contract duration, the highest-risk group is customers with 1.5 to 4 years of service; after that, churn becomes unlikely.

### Correlation Analysis

In [ ]:
# create a list of numeric columns
interval_cols = [
    'MonthlyCharges',
    'TotalCharges',
    'ContractDuration'
]

In [ ]:
# build the phik matrix
plt.figure(figsize=(10, 10))
sns.heatmap(main.phik_matrix(interval_cols=interval_cols), annot=True, fmt='.2f')
plt.show()

Goal of the correlation analysis: remove multicollinear features (>0.9) and features with very low correlation to the target (<0.05).

The 'gender' feature has correlation below the threshold, so it should be removed.

Multicollinearity exists among internet services and between 'MonthlyCharges' and 'InternetService'. Strong correlation between monthly charges and other internet services also exists but is below the threshold and may not be as problematic.
From internet services, keep only one feature that has the highest correlation with the target and the lowest with other features. That is 'OnlineBackup'.

To account for other internet services, it makes sense to create the 'NumInternetServices' feature.

In [ ]:
main = main.drop('gender', axis=1)

In [ ]:
main['NumInternetServices'] = (main[internet_columns] == 'Yes').sum(axis=1)

In [ ]:
internet_columns = internet_columns.drop('OnlineBackup')

In [ ]:
main = main.drop(internet_columns, axis=1)

In [ ]:
interval_cols.append('NumInternetServices')

In [ ]:
# build the phik matrix
plt.figure(figsize=(10, 10))
sns.heatmap(main.phik_matrix(interval_cols=interval_cols), annot=True, fmt='.2f')
plt.show()

The new feature has a notable correlation with the target (0.18) and is not multicollinear with other features, so adding it is reasonable.

### Overall Summary of EDA and Correlation Analysis

- Base and payments: month-to-month contracts dominate (~55%); paperless billing is used by 59%; the most common payment method is Electronic check (33.6%). IsBrokenContract is imbalanced: churn is 15.6%.
- Demographics: most customers are non-seniors (83.8%) and without dependents (70%); gender and partner status are nearly even.
- Money and customer value: MonthlyCharges varies widely (mean ~64.8; median ~70.35; 18.25-118.75). TotalCharges is right-skewed with outliers (up to ~9221), reflecting large differences in lifetime value/tenure.
- Services: 21.7% without internet and 9.7% without telephony. Fiber leads among internet connections (~44%), followed by DSL (~34.4%). Add-on internet services (backup/antivirus/support/parental control/streaming, etc.) are near yes/no with a tilt toward "do not use" for some options.
- Tenure (ContractDuration): the base consists of cohorts - many customers up to ~1 year (25% up to 276 days), a large cohort ~2 years (median 761 days ~ 2.1 years), and a notable share 4+ years (75% = 1461 days ~ 4 years, max ~ 6.3). Due to skewness, the median describes the "typical" customer better than the mean.

Class differences (associated with churn):
- Shorter contracts/month-to-month payment mean higher risk; two-year/one-year plans are more stable, and month-to-month is about half as frequent among churners (i.e., more strongly associated with churn).
- Electronic check is associated with a slightly higher churn share (~+5%).
- Men churn slightly more; seniors noticeably more.
- Partner is associated with higher churn share (almost 2x), Dependents has a weak effect.
- Fiber optic is associated with churn more often (~+5-7%); in general, having/using add-on internet/telephony services increases risk (possible proxy for dissatisfaction/complexity).
- For continuous features: higher MonthlyCharges means higher risk; for TotalCharges, risk is higher around ~2000-4000 and drops sharply after 4000 (near 0 above 8000 - the most loyal/long-tenured). For contract duration, risk is higher around 1.5-4 years, then becomes unlikely.

Correlation analysis (feature selection):
- Remove gender due to low correlation with the target (<0.05).
- Multicollinearity is detected among internet services and between MonthlyCharges and InternetService. To reduce redundancy, keep one most informative and least dependent internet service - OnlineBackup.
- To preserve information about service saturation, add NumInternetServices; it correlates with the target (~0.18) and does not create strong multicollinearity, so it is justified.

## Data Preparation

In [ ]:
# set values
RANDOM_STATE = 120626
TEST_SIZE = 0.25

In [ ]:
main['NumInternetServices'] = main['NumInternetServices'] .astype('float')

In [ ]:
# split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    main.drop('IsBrokenContract', axis=1),
    main['IsBrokenContract'],
    test_size = TEST_SIZE, 
    random_state = RANDOM_STATE,
    stratify=main['IsBrokenContract']
)

In [ ]:
# group dataframe columns by preprocessing method
num_columns = X_train.select_dtypes(include='float').columns.to_list()

In [ ]:
ohe_columns = X_train.drop(num_columns, axis=1).columns.to_list()

In [ ]:
ohe_columns

In [ ]:
data_preprocessor = ColumnTransformer(
    [('ohe', OneHotEncoder(drop='first', handle_unknown='error'), ohe_columns),
     ('num', StandardScaler(), num_columns)
    ]
)

**Summary:** In this step I created a preprocessing pipeline that accounts for feature types: numeric float features ('MonthlyCharges', 'TotalCharges', 'ContractDuration') are scaled, and the remaining features are encoded with standard one-hot encoding.

## Training Machine Learning Models

In this project we will train 4 models: 2 simple ones (logistic regression and decision tree) and 2 gradient boosting models (CatBoost and LGBM). We will train models separately to make it easier to find and fix issues. We will create a helper function find_the_best that performs a randomized search for the best model.

In [ ]:
# create a function to output the best model and metric value
def find_the_best(pipe, param_grid):
    _cv = KFold(n_splits=5, shuffle=True, random_state=42)
    randomized_search = RandomizedSearchCV(
    pipe,
    param_grid,
    cv=_cv,
    scoring='roc_auc',
    random_state=RANDOM_STATE,
    n_jobs=-1
    )
    randomized_search.fit(X_train, y_train)
    best_model = randomized_search.best_estimator_
    best_score = randomized_search.best_score_
    return {'pipe': randomized_search,
            'model': best_model, 
            'score': best_score,
            'params': randomized_search.best_params_}
        

### Logistic Regression

In [ ]:
# create a pipeline for logistic regression
pipe_logreg = Pipeline([
    ('preprocessor', data_preprocessor),
    ('model',  LogisticRegression(max_iter=3000, random_state=RANDOM_STATE)),
])

As hyperparameters, we will consider the regularization coefficient C and the maximum number of iterations, and also choose the best scaling method for numeric features.

In [ ]:
# parameter grid for logistic regression
param_grid_logreg =  {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__max_iter': [5000, 20000],
    'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler()]
    
}

In [ ]:
res_logreg_1 = find_the_best(pipe_logreg, param_grid_logreg)

In [ ]:
res_logreg_1['score']

In [ ]:
res_logreg_1['params']

It is worth considering wider hyperparameter ranges to avoid missing the best model.

In [ ]:
# create a new parameter dictionary
param_grid_logreg =  {
    'model__C': [0.1, 3],
    'model__max_iter': list(range(20000, 50000)),
    'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler()]
    
}

In [ ]:
res_logreg_2 = find_the_best(pipe_logreg, param_grid_logreg)

In [ ]:
res_logreg_2['score']

In [ ]:
res_logreg_2['params']

The metric value worsened slightly. It is likely that the first logistic regression model reached its best performance, but it still does not meet the target value (0.85).

### DecisionTreeClassifier Model

In [ ]:
# create a pipeline for decision tree
pipe_tree = Pipeline([
    ('preprocessor', data_preprocessor),
    ('model',  DecisionTreeClassifier(random_state=RANDOM_STATE)),
])

In [ ]:
# create a list of hyperparameters
param_grid_tree = {
    'model__max_depth': list(range(1, 7)),
    'model__min_samples_split': list(range(2, 7)),
    'model__min_samples_leaf': list(range(1, 7)),
    'preprocessor__num': [StandardScaler(), MinMaxScaler(), RobustScaler()]
}

In [ ]:
res_tree = find_the_best(pipe_tree, param_grid_tree)

In [ ]:
res_tree['score']

In [ ]:
res_tree['params']

The best model's hyperparameters are not at the ends of the range, so there is no need to expand them. The best metric is 0.78, which again does not reach the required threshold of 0.85.

### CatBoost Model

In [ ]:
# create a pipeline for categorical boosting
pipe_cat = Pipeline([
    ('preprocessor', data_preprocessor),
    ('model', CatBoostClassifier(early_stopping_rounds=200, random_seed=RANDOM_STATE, verbose=0)),
])

Categorical boosting does not require feature scaling, so we set 'preprocessor__num' to ['passthrough'] to avoid unnecessary search time.

In [ ]:
# create a hyperparameter list for categorical boosting
param_grid_cat = {
    'model__iterations': [500, 1500, 3000],
    'model__learning_rate': [0.03, 0.07, 0.1],
    'model__depth': [4, 6, 8],
    'preprocessor__num': ['passthrough']
}

In [ ]:
res_cat_1 = find_the_best(pipe_cat, param_grid_cat)

In [ ]:
res_cat_1['score']

In [ ]:
res_cat_1['params']

In [ ]:
param_grid_cat = {
    'model__iterations': range(1000, 2000, 100),
    'model__learning_rate': np.linspace(0.05, 1, 10),
    'model__depth': range(3, 7),
}

In [ ]:
res_cat_2 = find_the_best(pipe_cat, param_grid_cat)

In [ ]:
res_cat_2['score']

In [ ]:
res_cat_2['params']

### LGBM Model

In [ ]:
# create a pipeline for the LGBM model
pipe_lgbm = Pipeline([
    ('preprocessor', data_preprocessor),
    ('model', LGBMClassifier(random_state=RANDOM_STATE)),
])

The LGBM model, like categorical boosting, does not require scaling.

In [ ]:
# create a hyperparameter list for LGBM
param_grid_lgbm = {
    'model__n_estimators': [300, 800, 1500],
    'model__learning_rate': [0.03, 0.07, 0.1],
    'model__num_leaves': [31, 63, 127],
    'preprocessor__num': ['passthrough']
}

In [ ]:
res_lgbm_1 = find_the_best(pipe_lgbm, param_grid_lgbm)

In [ ]:
res_lgbm_1['score']

In [ ]:
res_lgbm_1['params']

In [ ]:
param_grid_lgbm = {
    'model__n_estimators': range(200, 500, 100),
    'model__learning_rate': np.arange(0.03, 1, 0.01),
    'model__num_leaves': range(20, 40, 5),
    'preprocessor__num': ['passthrough']
}

In [ ]:
res_lgbm_2 = find_the_best(pipe_lgbm, param_grid_lgbm)

In [ ]:
res_lgbm_2['score']

In [ ]:
res_lgbm_2['params']

The model reached the required value, but it is lower than the categorical boosting result.

### Best Model Selection

In [ ]:
score_dict = {
    'Logistic regression': res_logreg_1['score'],
    'Decision tree': res_tree['score'],
    'Categorical boosting': res_cat_1['score'],
    'LGBM model': res_lgbm_2['score']
}

**Summary:** The best quality was obtained after hyperparameter tuning of the CatBoost model with parameters {'preprocessor__num': 'passthrough',
 'model__learning_rate': 0.07,
 'model__iterations': 3000,
 'model__depth': 4}

## Best Model Predictions and Quality Evaluation

### Model Predictions and Metric Calculation

In [ ]:
# assign the best model to the model variable
model = res_cat_1['model']

In [ ]:
# predict class probabilities
y_pred_proba = model.predict_proba(X_test)[:, 1]

In [ ]:
# compute roc_auc
roc_auc_score(y_test, y_pred_proba)

The target metric is above the required threshold of 0.85 and even above the training value. It is worth checking the model's sanity and feature importance to be confident in this result.

### Model Sanity Check

In [ ]:
# create DummyClassifier models with different strategies and compute roc_auc for each
models = {
    "stratified": DummyClassifier(strategy="stratified"),
    "most_frequent": DummyClassifier(strategy="most_frequent"),
    "uniform": DummyClassifier(strategy="uniform")
}

for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred_dummy = clf.predict_proba(X_test)[:, 1]
    print(roc_auc_score(y_test, y_pred_dummy))

The metric values for all dummy models are around 0.5, so the model passes the sanity check.

### Feature Importance Analysis

In [ ]:
preprocessor = model['preprocessor']

In [ ]:
feature_names = preprocessor.get_feature_names_out()

In [ ]:
X_test_transformed = preprocessor.transform(X_test)

In [ ]:
pool_test = Pool(X_test_transformed)

shap_matrix = model['model'].get_feature_importance(pool_test, type="ShapValues")
shap_values = shap_matrix[:, :-1]   
base_value  = shap_matrix[:, -1]

In [ ]:
shap.summary_plot(
shap_values,
X_test_transformed,
feature_names=feature_names,
max_display=20,
plot_size=(16, 8),
show=False  
)

ax = plt.gca()
ax.set_title("Feature importance (SHAP)")
ax.set_ylabel("Feature names")

plt.tight_layout()
plt.show()

**Summary:**
1) The main protective factor is contract duration
- Long tenure (red points on the left) significantly reduces churn risk.
- Short tenure (blue points on the right) increases risk.
Conclusion: onboarding and retention in the first months are critical.

2) High monthly charges increase risk
- Red points are mostly on the right -> higher monthly charge pushes toward churn more often.
Conclusion: expensive plans need targeted offers/value/discounts, especially for low-tenure customers.

3) num__TotalCharges shows mixed, segment-based impact
- Low values often reduce risk (blue on the left).
- Blue points also appear on the right, which is associated with churn risk; this may correspond to low-tenure customers with low total charges.
- Very high values for some customers sharply increase risk (red tail on the right).
  Conclusion: TotalCharges should not be interpreted monotonically; analyze it together with ContractDuration and MonthlyCharges.

4) Contract type (one-hot ohe__Type_One year, ohe__Type_Two year)
- Long contract types look like risk-increasing factors (points shifted left), and their absence decreases risk. At first glance this looks illogical, but the same pattern appeared in EDA. Most likely, long contracts are unattractive due to termination conditions and "scare" customers.
  Conclusion: reconsider long-contract terms - increase discounts, simplify termination conditions, and make them beneficial and pleasant to use.

5) Socio-demographic features (partner/dependents, senior status)
- Senior status has a small effect but is more often associated with higher churn risk.
- Having a partner increases churn risk, which may indicate a lack of family-friendly subscriptions and use of competitors with such offers.
- Dependents have a nonlinear effect, difficult to evaluate.
Conclusion: consider social tariffs for seniors with discounts and family subscriptions.

6) Payment method
- Automatic payments have a small effect but are more often associated with churn than other methods.

### Accuracy, Confusion Matrix, and Interpretation

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
accuracy_score(y_test, y_pred)

Accuracy = 0.94, which means the model predicts the class correctly 94% of the time.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.show()


The model recognizes class 0 very well: few errors (15 FP with 1470 correct).
- For class 1 the situation is worse: there is a noticeable number of misses (84 FN), meaning the model often "does not see" ones and assigns them to 0.
- Therefore, despite high overall accuracy, the model performs better on the majority class (0), and quality on class 1 is limited.

The customer should compare the cost of losing clients with the cost of additional promos; it may make sense to improve recall. Currently the model favors saving promo budget by avoiding false positives for class 1.

## Overall Conclusion:

In this work I built a model that helps the telecom operator TeleDom assess churn risk in advance - that is, predict whether a subscriber will terminate the contract or stay. The data came from multiple sources, so the first important result was bringing them into a unified form: I properly handled dates, resolved nonstandard values (e.g., No in EndDate), created the target feature IsBrokenContract, converted numeric fields to correct types, and addressed blanks in TotalCharges (these were new customers as of the 2020-02-01 snapshot, so it was logical to replace with 0). I also checked for duplicates and found none.

After merging tables by customerID, I created a key engineered feature ContractDuration (contract length) and removed fields that either carry no meaning on their own or cause target leakage (e.g., EndDate). Missing values after merging were interpreted correctly: if a service cannot be used without internet/telephony, a missing value means "service not used", so I filled them as No internet service / No phone service.

Next I performed EDA on the combined dataset and confirmed several expected patterns: the target class is highly imbalanced (~15.6% churn), and the strongest features are in behavior/product areas - contract type, payment habits, plan price, connected services, and tenure. At the same time I checked correlations and reduced noise: removed gender as a weakly related feature and handled multicollinearity among internet services, keeping a more informative part (e.g., OnlineBackup) and adding NumInternetServices to preserve the idea of service saturation without inflating feature space.

Data preparation was organized in a pipeline: numeric features were scaled, categorical features were encoded (one-hot). This allows correct model training without manual workarounds and improves reproducibility.

I compared baseline models (logistic regression, decision tree) and stronger gradient boosting models (CatBoost and LGBM). The best result came from CatBoost after hyperparameter tuning (learning_rate=0.07, iterations=3000, depth=4): the target metric was about 0.916 in tuning and about 0.92 on the test set, above the required threshold of 0.85. I also checked sanity via dummy models (around 0.5), which showed the model is learning patterns rather than guessing.

I also analyzed feature importance and derived conclusions that can be used directly in business:
- The strongest protective factor is customer tenure: the longer the relationship, the lower the risk, and the first months/year are critical for retention.
- Higher MonthlyCharges increase risk - expensive plans need clearer value propositions and targeted retention offers, especially for newer customers.
- TotalCharges is not monotonic and works better as a segment indicator; it should not be interpreted separately from ContractDuration and MonthlyCharges.
- Contract type provides an important signal; my analysis suggests revisiting long-term contracts in terms of conditions, benefits, and termination convenience so they do not look risky to customers.
- Socio-demographic features (senior status, partner/dependents) add effects and can inform product decisions like family plans and social tariffs.

The confusion matrix also shows an important limitation: the model confidently recognizes class "stay" but misses some churners (notable FN). In other words, the current setup saves promo budget (fewer false alarms) but does not maximize recall for churners. In real deployment, I would recommend deciding what is more expensive for the business - losing a customer or giving an extra discount - and then adjusting the classification threshold / optimizing recall for class 1.

Overall, the result is a working and adequate model with quality above the target, clear interpretation of key churn factors, and practical retention recommendations: focus on early tenure, on customers with high monthly charges, and on revising offers/terms where product parameters statistically push customers toward termination.


**BUSINESS RECOMMENDATIONS:**

Focus on retention in the first months/first year
- Build an onboarding sequence: a series of touchpoints in the first 7/30/90 days (service quality check, setup help, useful service offers).
- Add risk triggers for new customers: if a customer is on a month-to-month plan with high MonthlyCharges, automatically send a personal offer/call.
- Track complaints/support contacts during this period (if available, this is a strong signal).

For expensive plans - targeted offers instead of mass discounts
- For customers with high MonthlyCharges, offer "value bundles": not just a discount, but bonuses that increase perceived value (additional services, faster internet, extended support).
- Test a "price guarantee" for 3-6 months or a temporary discount at churn risk (time-limited is better).
- Segment offers by tenure: expensive plan + short tenure = retention priority #1.

Revisit long-term contract logic (1-2 years)

Since there is a signal that long contracts may be perceived negatively/as a "trap", I would suggest:
- Make termination conditions clearer and softer (e.g., no complex penalties, clear formulas).
- Strengthen incentives: noticeable discount vs month-to-month, renewal bonuses, a "gift" for switching to 1-2 years.
- Change messaging from "sign up for 2 years" to "lock in terms and get value", i.e., sell it as risk reduction for the customer.

Work with payment habits
- For customers with manual/unstable payment methods (including electronic check), gently migrate to autopay: bonus for enabling, reminders, simplified process.
- Important: autopay is not a "cure" by itself, but it reduces friction and can lower spontaneous churn (when customers leave because they "got tired" or missed payment).

Segmentation by value and tenure (TotalCharges + ContractDuration)
- High-value, long-tenure customers should not be flooded with discounts without reason - better service, priority support, and individual terms.
- Medium-tenure customers with higher risk (about 1.5-4 years) should be re-engaged periodically with value: loyalty bonuses, plan refresh, upgraded terms.

Product bundles: family and social
Given the signal for Partner/partly SeniorCitizen, I would test:
- family plans (2-3 numbers/services cheaper in a bundle, shared bill, shared bonus);
- a "social" tariff for seniors (simple terms, clear discount, priority support).
  This can reduce churn risk by providing a fair price and a "right-fit" offer.

Operational recommendation: work not only with the model, but also with the threshold
  The current model avoids false positives but may miss some churners (FN). For business this means:
- tune the decision threshold based on economics: cost of losing a customer vs cost of a promo;
- create two modes:
  - "low-cost retention" (SMS/push/small bonus) - applied to a broader segment (higher recall);
  - "high-cost retention" (personal manager/substantial discount) - only for top-risk customers (higher precision).